In [0]:
from pyspark.sql.functions import col, count

SILVER_TABLE = "workspace.default.silver_hvfhv_trips"

df_silver = spark.table(SILVER_TABLE)

In [0]:
display(
    df_silver
    .groupBy("dispatching_base_num")
    .agg(
        count("*").alias("trip_count")
    )
    .orderBy(col("trip_count").desc())
)

In [0]:
display(
    df_silver
    .groupBy("originating_base_num")
    .agg(
        count("*").alias("trip_count")
    )
    .orderBy(col("trip_count").desc())
)

In [0]:
dispatch_bases = (
    df_silver
    .select(
        col("dispatching_base_num").alias("base_num")
    )
    .filter(col("base_num").isNotNull())
    .distinct()
)

originating_bases = (
    df_silver
    .select(
        col("originating_base_num").alias("base_num")
    )
    .filter(col("base_num").isNotNull())
    .distinct()
)

display(
    dispatch_bases
    .join(
        originating_bases,
        "base_num",
        "inner"
    )
    .orderBy("base_num")
)

In [0]:
from pyspark.sql.functions import col

SILVER_TABLE = "workspace.default.silver_hvfhv_trips"
DISPATCH_TABLE = "workspace.default.dim_dispatch"

df_silver = spark.table(SILVER_TABLE)

In [0]:
dispatching_bases = (
    df_silver
    .select(
        col("dispatching_base_num").alias("base_num")
    )
    .filter(col("base_num").isNotNull())
)

originating_bases = (
    df_silver
    .select(
        col("originating_base_num").alias("base_num")
    )
    .filter(col("base_num").isNotNull())
)

df_dispatch = (
    dispatching_bases
    .union(originating_bases)
    .distinct()
)

In [0]:
df_dispatch = (
    df_dispatch
    .withColumn(
        "dispatch_key",
        col("base_num")
    )
    .select(
        "dispatch_key",
        "base_num"
    )
)

In [0]:
print("Dimension rows:", df_dispatch.count())
print("Dimension columns:", len(df_dispatch.columns))

display(
    df_dispatch.orderBy("base_num")
)

In [0]:
print("Dimension rows:", df_dispatch.count())
print("Dimension columns:", len(df_dispatch.columns))

display(
    df_dispatch.orderBy("base_num")
)

In [0]:
display(
    df_dispatch
    .groupBy("dispatch_key")
    .count()
    .filter(col("count") > 1)
)

In [0]:
print(
    "Null base numbers:",
    df_dispatch.filter(col("base_num").isNull()).count()
)

In [0]:
(
    df_dispatch
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(DISPATCH_TABLE)
)

In [0]:
df_dispatch_saved = spark.table(DISPATCH_TABLE)

print("Persisted rows:", df_dispatch_saved.count())
print("Persisted columns:", len(df_dispatch_saved.columns))

display(
    df_dispatch_saved.orderBy("base_num")
)

In [0]:
from pyspark.sql.functions import col

DISPATCH_TABLE = "workspace.default.dim_dispatch"

# Get all dispatch/originating bases used by February
df_feb_dispatch = (
    spark.table("workspace.default.silver_hvfhv_trips")
    .filter(col("_source_month") == "2026-02")
    .select(
        "dispatching_base_num",
        "originating_base_num"
    )
)

feb_bases = (
    df_feb_dispatch
    .selectExpr("dispatching_base_num as base_num")
    .union(
        df_feb_dispatch
        .selectExpr("originating_base_num as base_num")
    )
    .filter(col("base_num").isNotNull())
    .distinct()
)

# Read existing dispatch dimension
existing_bases = (
    spark.table(DISPATCH_TABLE)
    .select("base_num")
)

# Combine existing + February bases
all_bases = (
    existing_bases
    .union(feb_bases)
    .distinct()
)

# Create updated dimension
df_dispatch_updated = (
    all_bases
    .withColumn(
        "dispatch_key",
        col("base_num")
    )
    .select(
        "dispatch_key",
        "base_num"
    )
    .orderBy("base_num")
)

display(df_dispatch_updated)

In [0]:
(
    df_dispatch_updated
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DISPATCH_TABLE)
)

In [0]:
print(
    "Dispatch dimension rows:",
    spark.table(DISPATCH_TABLE).count()
)